In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from datetime import datetime

In [5]:
dataset = "impli"

sim_functions = [
    "cos",
    "cka",
    "wasser",
]

agg_functions = [
    "mean",
    "max",
    "sum",
    "entropy",
    "gini"
]


model_map = {
    "google-bert/bert-base-uncased": "BERT-base Uncased",
    "google-bert/bert-base-cased": "BERT-base Cased",
    "google-bert/bert-large-uncased": "BERT-large Uncased",
    "google-bert/bert-large-cased": "BERT-large Cased",
    "answerdotai/ModernBERT-base": "ModernBERT-base",
    "answerdotai/ModernBERT-large": "ModernBERT-large",
}

models = list(model_map.keys())

layer_map = {
    "google-bert/bert-base-uncased": list(range(12)),
    "google-bert/bert-base-cased": list(range(12)),
    "google-bert/bert-large-uncased": list(range(24)),
    "google-bert/bert-large-cased": list(range(24)),
    "answerdotai/ModernBERT-base": list(range(22)),
    "answerdotai/ModernBERT-large": list(range(28)),
}

# master file
# ROOT = Path.cwd().parent

# scores_dir = ROOT / "data" / "processed" 
csv_file = f"../../data/processed/impli_layers.csv"

print(csv_file)

../../data/processed/impli_layers.csv


In [6]:
df = pd.read_csv(csv_file)

df.shape

(964410, 18)

## ranked corr(model decomp, syntax flexibility)

In [7]:
import pandas as pd
from scipy.stats import spearmanr

#syntax file
df_syntax = pd.read_csv("/Users/mmi/Documents/projects/idioms_decomposability/decomp_code/idioms_decomposability/data/total_entropy.csv")


column_to_merge = "premise"

merged = df.merge(
    df_syntax,
    on=column_to_merge,
    how="inner"
)

# merged.shape
merged.columns

Index(['Unnamed: 0_x', 'premise', 'hypothesis_x', 'extracted_idiom_x',
       'base_form_x', 'idiom_pos_x', 'idiom_processed', 'token_scores',
       'decomp_score', 'sim_function', 'model_name', 'model_label',
       'model_folder', 'timestamp', 'layer', 'sim_func', 'agg_func',
       'source_path', 'Unnamed: 0_y', 'hypothesis_y', 'extracted_idiom_y',
       'base_form_y', 'idiom_pos_y', 'identity_cql', 'passive_cql',
       'adj_insertion_cql_1', 'adj_insertion_cql_2', 'adv_insertion_cql',
       'adv_insertion_cql_adj', 'nominalization_cql', 'adj_insertion_full_1',
       'adj_insertion_full_2', 'adj_insertion_rel_1', 'adj_insertion_rel_2',
       'adv_insertion_full', 'adv_insertion_full_adj', 'adv_insertion_rel',
       'adv_insertion_rel_adj', 'identity_full', 'identity_rel',
       'nominalization_full', 'nominalization_rel', 'passive_full',
       'passive_rel', 'entropy_full', 'entropy_rel'],
      dtype='object')

In [8]:
def run_correlation_df(
    df,
    x_col="entropy_full",
    y_col="decomp_score"
):
    # drop NaNs just in case
    sub = df[[x_col, y_col]].dropna()

    if len(sub) < 3:
        return None

    corr, p_value = spearmanr(sub[x_col], sub[y_col])

    return {
        "spearmanr": corr,
        "p_value": p_value,
        "n": len(sub),
    }


In [9]:
group_cols = [
    "model_name",
    "layer",
    "sim_func",
    "agg_func",
    "timestamp",
]

results = []

for keys, group in merged.groupby(group_cols):
    stats = run_correlation_df(group)

    if stats is None:
        continue

    record = dict(zip(group_cols, keys))
    record.update(stats)
    results.append(record)

corr_df = pd.DataFrame(results)
corr_df.head()




,model_name,layer,sim_func,agg_func,timestamp,spearmanr,p_value,n
0,answerdotai/ModernBERT-base,0,cka,entropy,2025-12-25_02:34:31,-0.084239,0.053277,527
1,answerdotai/ModernBERT-base,0,cka,gini,2025-12-25_02:34:31,-0.043902,0.314456,527
2,answerdotai/ModernBERT-base,0,cka,max,2025-12-25_02:34:31,-0.009581,0.826316,527
3,answerdotai/ModernBERT-base,0,cka,mean,2025-12-25_02:34:31,0.000230,0.995805,527
4,answerdotai/ModernBERT-base,0,cka,sum,2025-12-25_02:34:31,-0.022684,0.603364,527


## Significant only

In [10]:
sig = corr_df[corr_df["p_value"] < 0.05]

print(sig.shape)
sig.head()

(443, 8)


,model_name,layer,sim_func,agg_func,timestamp,spearmanr,p_value,n
11,answerdotai/ModernBERT-base,0,wasser,gini,2025-12-25_02:34:31,-0.134499,0.001972,527
15,answerdotai/ModernBERT-base,1,cka,entropy,2025-12-25_02:34:31,-0.095315,0.028678,527
25,answerdotai/ModernBERT-base,1,wasser,entropy,2025-12-25_02:34:31,-0.098226,0.024132,527
26,answerdotai/ModernBERT-base,1,wasser,gini,2025-12-25_02:34:31,-0.121115,0.005369,527
30,answerdotai/ModernBERT-base,2,cka,entropy,2025-12-25_02:34:31,-0.090483,0.037847,527


In [11]:
sig = sig.sort_values(by="spearmanr")

print(sig.shape)

sig.head(10)

(443, 8)


,model_name,layer,sim_func,agg_func,timestamp,spearmanr,p_value,n
311,answerdotai/ModernBERT-base,20,wasser,gini,2025-12-25_02:34:31,-0.161356,0.000199,527
1661,google-bert/bert-large-uncased,12,wasser,gini,2025-12-25_02:34:13,-0.144657,0.000867,527
1436,google-bert/bert-large-cased,21,wasser,gini,2025-12-25_02:34:13,-0.141850,0.001094,527
854,google-bert/bert-base-cased,6,wasser,sum,2025-12-25_02:34:13,-0.139946,0.001278,527
1706,google-bert/bert-large-uncased,15,wasser,gini,2025-12-25_02:34:13,-0.139756,0.001298,527
1166,google-bert/bert-large-cased,3,wasser,gini,2025-12-25_02:34:13,-0.134828,0.001922,527
836,google-bert/bert-base-cased,5,wasser,gini,2025-12-25_02:34:13,-0.134793,0.001927,527
11,answerdotai/ModernBERT-base,0,wasser,gini,2025-12-25_02:34:31,-0.134499,0.001972,527
881,google-bert/bert-base-cased,8,wasser,gini,2025-12-25_02:34:13,-0.134467,0.001977,527
1051,google-bert/bert-base-uncased,8,cka,gini,2025-12-25_02:34:13,-0.133360,0.002155,527


In [12]:
sig['model_name'].value_counts()

model_name
google-bert/bert-large-cased      115
google-bert/bert-large-uncased     91
answerdotai/ModernBERT-large       66
google-bert/bert-base-uncased      61
google-bert/bert-base-cased        58
answerdotai/ModernBERT-base        52
Name: count, dtype: int64

In [40]:
sig["spearmanr"] = sig["spearmanr"].round(3)
sig["p_value"] = sig["p_value"].round(3)

## large models's effect sizes

In [41]:
sig_bert_large_cased = sig[sig['model_name'] == "google-bert/bert-large-cased"].sort_values(by="spearmanr").drop(columns=["timestamp"], inplace=False)


sig_bert_large_cased.head()
print(sig_bert_large_cased.shape)


sig_bert_large_cased.head(5).to_csv("../model_decomp_vs_syntax_large_models/bert_large_cased.csv", index=False)

(115, 7)


In [42]:
sig_bert_large_uncased = sig[sig['model_name'] == "google-bert/bert-large-uncased"].sort_values(by="spearmanr").drop(columns=["timestamp"], inplace=False)

sig_bert_large_uncased.head()
print(sig_bert_large_uncased.shape)

sig_bert_large_uncased.head(5).to_csv("../model_decomp_vs_syntax_large_models/bert_large_uncased.csv", index=False)

(91, 7)


In [43]:
sig_modernBert_large = sig[sig['model_name'] == "answerdotai/ModernBERT-large"].sort_values(by="spearmanr").drop(columns=["timestamp"], inplace=False)

sig_modernBert_large.head()
print(sig_modernBert_large.shape)

sig_modernBert_large.head(5).to_csv("../model_decomp_vs_syntax_large_models/modernBert_large.csv", index=False)

(66, 7)
